# RocketPy — Full Pipeline (Flight + Hot-fire + SMT0033)

This notebook replicates the original colab `rocketpy_flight_simulation.ipynb` and adds the full data-processing history:

- retiming onboard time using Ptank correlation
- Pc->Thrust calibration
- flight thrust reconstruction (**no ignition synthesis**)
- Flight vs SMT0033 correlation plots
- export a RASP `.eng` and run RocketPy

**No ignition synthesis** = do not copy HFT4 ignition shape into the flight curve.


In [ ]:
!pip install -q rocketpy numpy matplotlib pandas openpyxl


In [ ]:
# --- Colab / repo setup (fix ModuleNotFoundError) ---
import os, sys, subprocess

def sh(cmd):
    print('+', cmd)
    subprocess.check_call(cmd, shell=True)

if not os.path.exists('scripts/flight_thrust_pipeline.py'):
    # When opened from GitHub in Colab, the repository is not automatically cloned.
    if not os.path.exists('hybrid-rocket-trajectory'):
        sh('git clone https://github.com/jmartos-br/hybrid-rocket-trajectory.git')
    os.chdir('hybrid-rocket-trajectory')
    sh('git checkout rocketpy-retimed-motor')

# Ensure repo root is importable
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print('CWD:', os.getcwd())
print('Has scripts/flight_thrust_pipeline.py:', os.path.exists('scripts/flight_thrust_pipeline.py'))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from rocketpy import Environment, Rocket, Flight, GenericMotor


## Upload private flight logs (Colab)

This repository may **not** include the raw flight logs (private). If you are running on **Google Colab**, upload them here.

**Required** (flight onboard, semicolon `;` separated):
- `telemetry_log_2_cut.csv` (or equivalent) with columns compatible with the loader (`Time`, `PT`, `PC`).

**Optional** (transmitted telemetry, semicolon `;` separated):
- `Pressure-Temperature-Flight-Data_flight2.csv` with columns `timestamp`, `pt_pressure`, `pc_pressure`.

After uploading, the notebook will auto-pick the uploaded filenames if the default paths are missing.


In [ ]:
# Colab upload helper (safe to run multiple times)
import os

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('IN_COLAB:', IN_COLAB)

if IN_COLAB:
    print('Upload the private flight logs now (onboard required; telemetry optional).')
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))
else:
    print('Not running in Colab. Ensure files exist under data/ or set ONBOARD_PATH/TELEM_PATH manually.')


## Import helpers


In [ ]:
from scripts.flight_thrust_pipeline import (
    load_flight_onboard_semicolon,
    load_telemetry_semicolon,
    load_smt_extracted_csv,
    retime_onboard_by_ptank,
    calibrate_pc_to_thrust_origin,
    calibrate_pc_to_thrust_intercept,
    reconstruct_thrust_from_pc,
    export_rasp_eng,
)


## 1) Load datasets


In [ ]:
# --- Input paths ---
# NOTE: The flight onboard log may be private and not committed to the repo.
ONBOARD_PATH = 'data/telemetry_log_2_cut.csv'   # onboard (semicolon)
TELEM_PATH   = 'data/Pressure-Temperature-Flight-Data_flight2.csv'  # optional telemetry
SMT_PATH     = 'data/SMT0033_extracted.csv'

print('Repo data/ contains:', sorted(os.listdir('data')) if os.path.exists('data') else 'NO data/ folder')

# --- Load SMT0033 (public in repo) ---
smt = load_smt_extracted_csv(SMT_PATH)

# --- Auto-pick uploaded files (Colab) if defaults are missing ---
uploaded = globals().get('uploaded', {}) or {}

def _pick_uploaded(candidates_substrings):
    keys = list(uploaded.keys())
    for sub in candidates_substrings:
        for k in keys:
            if sub.lower() in k.lower():
                return k
    return None

if not os.path.exists(ONBOARD_PATH):
    cand = _pick_uploaded(['telemetry_log', 'onboard'])
    if cand:
        ONBOARD_PATH = cand
        print('Using uploaded onboard file:', ONBOARD_PATH)
    else:
        raise FileNotFoundError(f"Missing {ONBOARD_PATH}. In Colab: run the upload cell, upload the onboard semicolon CSV, then re-run this cell.")

if not os.path.exists(TELEM_PATH):
    cand = _pick_uploaded(['Pressure-Temperature-Flight-Data', 'flight2', 'telemetry'])
    if cand and cand != ONBOARD_PATH:
        TELEM_PATH = cand
        print('Using uploaded telemetry file:', TELEM_PATH)

# --- Load flight onboard (required) ---
onboard = load_flight_onboard_semicolon(ONBOARD_PATH)

# --- Select reference for retiming (telemetry if present, else SMT0033) ---
use_telem = os.path.exists(TELEM_PATH)
if use_telem:
    telem = load_telemetry_semicolon(TELEM_PATH)
    ref = telem
    ref_name = 'telemetry'
else:
    ref = smt
    ref_name = 'smt0033'

print('Onboard N:', len(onboard.t))
print('Reference:', ref_name)


### Scientific logic: ignition alignment vs retiming vs calibration

This notebook uses **three different concepts** that must not be mixed:

1) **Ignition alignment (event definition)** -- we define $$t=0$$ when **chamber pressure** exceeds a threshold:
   - ignition when **Pc > 2 bar** (same rule for flight, telemetry, and SMT0033).

2) **Retiming (fix the onboard clock)** -- the onboard log time axis can be compressed/non-physical. We correct it using **tank pressure (Ptank)** as an external *clock proxy* because:
   - Ptank is **monotonic during blowdown** and present in flight + reference
   - Ptank is **smoother** than Pc and less sensitive to injector/combustion dynamics
   - we only use Ptank to compute a **time scale factor**, not to compute thrust.

3) **Thrust calibration & reconstruction** -- thrust is calibrated/reconstructed from **chamber pressure**, not from Ptank:
   - Calibrate **Pc_gauge = Pc - Pc_baseline** against **measured thrust** (SMT0033).
   - Apply calibration to **flight Pc_gauge** to reconstruct thrust.

So: **Ptank is used only to fix time**, while **Pc is used to define ignition and to reconstruct thrust**.


## 2) Retiming the onboard timebase (using Ptank as a clock)

**Step A -- ignition alignment:** First we shift each dataset so that $$t=0$$ is the first sample where **Pc > 2 bar**.

**Step B -- retiming:** We compute a single scale factor `scale` such that:

$$ t_{real} \approx scale \cdot t_{onboard} $$

We estimate `scale` by matching **times at the same Ptank levels** (anchors) between onboard and the reference.

Why Ptank? It behaves as a monotonic blowdown state variable and is less sensitive than Pc to combustion transients.

Important: **This does not calibrate thrust**. It only fixes time.


In [ ]:
# Retiming: onboard -> reference using Ptank anchors
# Returns corrected time axis for onboard (seconds from ignition)
t_real, scale = retime_onboard_by_ptank(onboard, ref, pc_thresh=2.0, n_anchors=20)
print('Retiming scale (onboard -> reference):', scale)

# --- Diagnostic: ignition alignment by Pc>2 bar ---
# We show Pc vs time after shifting by the Pc threshold crossing (event definition only)
from scripts.flight_thrust_pipeline import align_by_pc

t_on_rel, _ = align_by_pc(onboard.t, onboard.pc, pc_thresh=2.0)
t_ref_rel, _ = align_by_pc(ref.t, ref.pc, pc_thresh=2.0)

plt.figure(figsize=(10,4))
plt.plot(t_on_rel, onboard.pc, 'o-', label='Onboard Pc (shifted)')
plt.plot(t_ref_rel, ref.pc, 'o-', label=f'Reference Pc (shifted: {ref_name})', alpha=0.7)
plt.axvline(0, color='k', lw=1)
plt.xlim(-1, 5)
plt.xlabel('Time from ignition by Pc>2 bar [s]')
plt.ylabel('Pc [bar]')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Ignition alignment definition (Pc>2 bar)')
plt.show()

# --- Diagnostic: Ptank overlap after retiming ---
plt.figure(figsize=(10,5))
plt.plot(t_real, onboard.ptank, 'o-', label='Onboard Ptank (retimed)')
plt.plot(t_ref_rel, ref.ptank, 'o-', label=f'Reference Ptank ({ref_name})', alpha=0.7)
plt.xlim(-2, max(t_real.max(), t_ref_rel.max()) + 2)
plt.xlabel('Time from ignition [s]')
plt.ylabel('Tank pressure Ptank [bar]')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Retiming check: Ptank alignment')
plt.show()


## 3) Pc_gauge -> Thrust calibration (using SMT0033 measured thrust)

Here we compute a calibration between **chamber pressure (Pc)** and **measured thrust** from SMT0033.

- We remove baseline: $$Pc_{gauge} = \max(Pc - Pc_0, 0)$$
- Fit either:
  - **Origin fit**: $$F = a\,Pc_{gauge}$$ (physically enforces $$F=0$$ when $$Pc_{gauge}=0$$)
  - **Intercept fit**: $$F = a\,Pc_{gauge} + b$$

Again: **Ptank is not used to compute thrust** -- only Pc is used here.


In [ ]:
# Compute Pc baseline from pre-ignition and build Pc_gauge
pc0_smt = np.median(smt.pc[smt.t < 0]) if np.any(smt.t < 0) else np.median(smt.pc[:50])
pcg_smt = np.maximum(smt.pc - pc0_smt, 0)

# Use a mask to avoid very early transient points for calibration
mask = (smt.t > 0.5) & (pcg_smt > 1.0) & (smt.thrust_N > 5)

cal_origin = calibrate_pc_to_thrust_origin(pcg_smt[mask], smt.thrust_N[mask])
cal_inter  = calibrate_pc_to_thrust_intercept(pcg_smt[mask], smt.thrust_N[mask])

print('Origin fit:', cal_origin)
print('Intercept fit:', cal_inter)

# Visual check: thrust vs Pc_gauge
plt.figure(figsize=(7,5))
plt.plot(pcg_smt[mask], smt.thrust_N[mask], 'o', ms=3, alpha=0.6, label='SMT0033 (measured)')

x = np.linspace(0, max(1.0, float(np.max(pcg_smt[mask]))), 200)
plt.plot(x, cal_origin.slope_N_per_bar * x, '-', label=f'Origin fit: a={cal_origin.slope_N_per_bar:.3f} N/bar')
plt.plot(x, cal_inter.slope_N_per_bar * x + cal_inter.intercept_N, '--', label=f'Intercept fit')

plt.xlabel('Pc_gauge [bar]')
plt.ylabel('Thrust [N]')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Calibration check: Pc_gauge -> Thrust')
plt.show()


## 4) Reconstruct flight thrust (NO ignition synthesis)


In [ ]:
pc0_f = np.median(onboard.pc[t_real < 0]) if np.any(t_real < 0) else np.median(onboard.pc[:10])
cal = cal_origin
flight_thrust = reconstruct_thrust_from_pc(onboard.pc, cal, pc0=pc0_f)

plt.figure(figsize=(10,5))
plt.plot(t_real, flight_thrust, 'o-')
plt.xlabel('Time from ignition [s]')
plt.ylabel('Thrust [N]')
plt.grid(alpha=0.3)
plt.title('Flight thrust (no ignition synthesis)')
plt.show()


## 5) Correlation: Thrust vs Tank Pressure (Flight vs SMT0033)


In [ ]:
mask_f = (t_real >= 0) & (flight_thrust > 1)
order_f = np.argsort(-onboard.ptank[mask_f])
mask_s = (smt.t >= 0) & (smt.thrust_N > 1)
order_s = np.argsort(-smt.ptank[mask_s])

plt.figure(figsize=(10,6))
plt.plot(smt.ptank[mask_s][order_s], smt.thrust_N[mask_s][order_s], '-', label='SMT0033 (measured)')
plt.plot(onboard.ptank[mask_f][order_f], flight_thrust[mask_f][order_f], '-', label='Flight (reconstructed)')
plt.gca().invert_xaxis()
plt.xlabel('Tank pressure [bar]')
plt.ylabel('Thrust [N]')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Thrust vs Tank Pressure')
plt.show()


## 6) Export `.eng` and run RocketPy


In [ ]:
OUT_ENG = 'Flight_Reconstructed_Retimed_NoIgn.eng'
idx = np.argsort(t_real)
t_sorted = t_real[idx]
f_sorted = flight_thrust[idx]
mask = t_sorted >= 0
te = np.append(t_sorted[mask], t_sorted[mask][-1] + 0.01)
fe = np.append(f_sorted[mask], 0.0)
export_rasp_eng(OUT_ENG, name='FlightRetimedNoIgn', t=te, thrust=fe)
print('Wrote', OUT_ENG)


In [ ]:
# Environment (from rocketpy_flight_simulation.ipynb)
env = Environment(latitude=24.18133, longitude=53.688379, elevation=5)
env.set_date((2026, 2, 13, 12))
env.set_atmospheric_model(
    type='custom_atmosphere',
    wind_u=[(0, 0.07), (10, 0.07), (135, 0.00), (818, -0.45), (1542, -0.62), (3164, -0.51), (5854, 12.69)],
    wind_v=[(0, -4.00), (10, -4.00), (135, -4.19), (818, -1.46), (1542, 1.40), (3164, -0.51), (5854, 4.62)],
    pressure=[(0, 101500), (135, 100000), (818, 92500), (1542, 85000), (3164, 70000), (5854, 50000)],
    temperature=[(0, 302.95), (135, 301.65), (818, 295.25), (1542, 288.75), (3164, 281.35), (5854, 264.25)],
)

burn_time = float(te[-2])
motor = GenericMotor(
    thrust_source=OUT_ENG,
    burn_time=burn_time,
    chamber_radius=0.05,
    chamber_height=1.33,
    chamber_position=1.33/2,
    propellant_initial_mass=2.42,
    nozzle_radius=0.025,
    dry_mass=6.9,
    dry_inertia=(0.5, 0.5, 0.01),
    nozzle_position=0.0,
    center_of_dry_mass_position=1.33/2,
    coordinate_system_orientation='nozzle_to_combustion_chamber',
)

rocket = Rocket(
    radius=0.05,
    mass=3.780,
    inertia=(3.5, 3.5, 0.005),
    power_off_drag='data/poweroff_drag.csv',
    power_on_drag='data/poweron_drag.csv',
    center_of_mass_without_motor=1.869,
    coordinate_system_orientation='tail_to_nose',
)
rocket.add_motor(motor, position=0.0)
rocket.add_nose(length=0.3, kind='ogive', position=2.600)
rocket.add_trapezoidal_fins(n=4, root_chord=0.145, tip_chord=0.065, span=0.08, sweep_length=0.11, cant_angle=0.5, position=0.145)
rocket.add_tail(top_radius=0.05, bottom_radius=0.03, length=0.055, position=0.0)
rocket.set_rail_buttons(upper_button_position=1.80, lower_button_position=0.40, angular_position=88)

def main_trigger(p, h, y):
    return True if y[5] < 0 else False

rocket.add_parachute(
    name='Main',
    cd_s=2.2 * np.pi * (1.8288/2)**2,
    trigger=main_trigger,
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

flight = Flight(rocket=rocket, environment=env, rail_length=7.0, inclination=83.5, heading=90, max_time=600, time_overshoot=True)
print('Apogee AGL [m]:', float(flight.apogee - env.elevation))


## Export results (optional)

If running on Colab, you can zip and download any generated `.eng` files and figures.


In [ ]:
import os, glob, zipfile

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

out_dir = 'outputs_colab'
os.makedirs(out_dir, exist_ok=True)

# Collect common artifacts
for p in glob.glob('*.eng') + glob.glob('*.png') + glob.glob('*.csv'):
    try:
        base = os.path.basename(p)
        if base == os.path.basename(__file__):
            continue
        # copy into outputs
        with open(p,'rb') as fsrc, open(os.path.join(out_dir, base),'wb') as fdst:
            fdst.write(fsrc.read())
    except Exception:
        pass

zip_path = 'outputs_colab.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for root, _, files_ in os.walk(out_dir):
        for fn in files_:
            full = os.path.join(root, fn)
            z.write(full, arcname=os.path.relpath(full, '.'))

print('Wrote', zip_path)
if IN_COLAB:
    files.download(zip_path)
